# 🚀 Giving Small Models a Second Wind: Anti-Self-Distillation (AntiSD) on Gemma 4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

This notebook implements **Anti-Self-Distillation (AntiSD)** from Shen et al. (2026) for Google DeepMind's **Gemma 4** (`google/gemma-4-e2b-it`).

### The Core Idea:
- **Student** (\( \pi_S \)): Generates reasoning traces inside `<think>...</think>`.
- **Self-Teacher** (\( \pi_T \)): The same model given a verified reference solution and feedback.
- **Shortcut Bias:** The teacher gets overly confident on final answer templates and penalizes intermediate hesitation.
- **AntiSD Fix:** Invert the gradient via Jensen-Shannon Divergence ascent with softplus bounding and an entropy gate.


In [ ]:
# 1. Install Dependencies
!pip install -q -U transformers peft datasets accelerate bitsandbytes torch

In [ ]:
# 2. Check GPU & Hardware
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# 3. Hugging Face Login (to access Gemma 4 weights)
from huggingface_hub import login
import os

# Pass your Hugging Face Token here if required
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("Please log in to Hugging Face if access to google/gemma-4-e2b-it requires authorization.")

In [ ]:
# 4. Define the 20-Line AntiSD Advantage Kernel
import math
import torch.nn.functional as F

def compute_antisd_advantage(student_logp, teacher_logp, gate_active=True):
    """
    Computes JSD-derived Anti-Self-Distillation advantage:
    u_t = t_t - s_t  (conditional Pointwise Mutual Information)
    A_t^{AntiSD} = -0.5 * (softplus(u_t) - log(2))
    """
    u_t = teacher_logp - student_logp
    antisd_adv = -0.5 * (F.softplus(u_t) - math.log(2.0))
    if not gate_active:
        antisd_adv = torch.zeros_like(antisd_adv)
    return antisd_adv

In [ ]:
# 5. Load Gemma 4 Model & Configure LoRA
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

MODEL_NAME = "google/gemma-4-e2b-it"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# For free Colab T4 (16GB), use 4-bit QLoRA. For A100/L4, load directly in bfloat16
use_4bit = torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory < 20e9
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16) if use_4bit else None

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 6. Load Dataset (GSM8K Math Benchmark)
from datasets import load_dataset
import re

dataset = load_dataset("gsm8k", "main", split="train")
print(f"Loaded GSM8K dataset: {len(dataset)} problems.")

def extract_number(text):
    if "####" in text:
        ans = text.split("####")[-1].strip().replace(",", "")
        m = re.search(r"[-+]?\d*\.?\d+", ans)
        if m: return float(m.group(0))
    nums = re.findall(r"[-+]?\d*\.?\d+", text.replace(",", ""))
    return float(nums[-1]) if nums else None

def compute_reward(pred_str, gold_str):
    p = extract_number(pred_str)
    g = extract_number(gold_str)
    return 1.0 if (p is not None and g is not None and abs(p - g) < 1e-4) else 0.0

In [ ]:
# 7. Run 50-Step AntiSD Training Loop
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)
TOTAL_STEPS = 50
WARMUP_STEPS = 5
GROUP_SIZE = 4
LAMBDA_ASD = 0.1

warmup_entropies = []
h_warm = None
tau_down = None
gate_open = True

print("Starting AntiSD 50-Step Post-Training...")
for step in range(TOTAL_STEPS):
    sample = dataset[step % len(dataset)]
    problem, ground_truth = sample["question"], sample["answer"]

    s_prompt = f"<start_of_turn>user\nSolve step-by-step. Show reasoning inside <think> tags.\n\n{problem}<end_of_turn>\n<start_of_turn>model\n<think>\n"
    
    # Generate rollouts
    model.eval()
    enc = tokenizer(s_prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outs = model.generate(
            **enc, max_new_tokens=384, temperature=0.7, num_return_sequences=GROUP_SIZE, do_sample=True, pad_token_id=tokenizer.eos_token_id
        )
    
    rollouts = [tokenizer.decode(outs[i][enc.input_ids.shape[1]:], skip_special_tokens=True) for i in range(GROUP_SIZE)]
    rewards = [compute_reward(r, ground_truth) for r in rollouts]
    r_t = torch.tensor(rewards, device=device, dtype=torch.float32)
    seq_adv = (r_t - r_t.mean()) / (r_t.std() + 1e-6)

    # Student Forward Pass
    model.train()
    s_inputs = tokenizer([s_prompt + r for r in rollouts], padding=True, return_tensors="pt").to(device)
    s_logits = model(**s_inputs).logits
    s_logp = F.log_softmax(s_logits[:, :-1], dim=-1).gather(-1, s_inputs.input_ids[:, 1:].unsqueeze(-1)).squeeze(-1)

    # Teacher Forward Pass (Privileged Context)
    t_prompts = [
        f"<start_of_turn>user\nSolve step-by-step inside <think> tags.\n\n{problem}\n\n[Verified Solution]\n{ground_truth}\nAssessment: {correct if r > 0.5 else incorrect}<end_of_turn>\n<start_of_turn>model\n<think>\n"
        for r in rewards
    ]
    t_inputs = tokenizer([tp + r for tp, r in zip(t_prompts, rollouts)], padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        t_logits = model(**t_inputs).logits
        t_logp = F.log_softmax(t_logits[:, :-1], dim=-1).gather(-1, t_inputs.input_ids[:, 1:].unsqueeze(-1)).squeeze(-1)
        
        # Compute median teacher entropy
        t_probs = F.softmax(t_logits, dim=-1)
        t_entropy = -(t_probs * F.log_softmax(t_logits, dim=-1)).sum(dim=-1)
        med_h = torch.median(t_entropy).item()

    # Gate calibration & Schmitt trigger
    if step < WARMUP_STEPS:
        warmup_entropies.append(med_h)
        cur_lambda = 0.0
    else:
        if h_warm is None:
            h_warm = float(torch.median(torch.tensor(warmup_entropies)).item())
            tau_down = 0.93 * h_warm
        if gate_open and med_h < tau_down: gate_open = False
        elif not gate_open and med_h >= h_warm: gate_open = True
        cur_lambda = LAMBDA_ASD if gate_open else 0.0

    # Compute AntiSD Advantage and Loss
    min_len = min(s_logp.shape[1], t_logp.shape[1])
    antisd_adv = compute_antisd_advantage(s_logp[:, :min_len], t_logp[:, :min_len], gate_active=(cur_lambda > 0))
    total_adv = seq_adv.unsqueeze(1) + cur_lambda * antisd_adv
    
    loss = -(total_adv.detach() * s_logp[:, :min_len]).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (step + 1) % 5 == 0:
        print(f"[Step {step+1:02d}/50] Loss: {loss.item():.4f} | Mean Reward: {r_t.mean().item():.2f} | Teacher H: {med_h:.3f} | Gate: {Open if gate_open else Closed}")

print("\nAntiSD Training Complete! 🎉")

In [ ]:
# 8. Test the Model Before vs After
test_problem = "A baker sells 12 loaves of bread in the morning and twice as many in the afternoon. If each loaf costs $4, how much did the baker earn?"
test_prompt = f"<start_of_turn>user\nSolve step-by-step. Show reasoning inside <think> tags.\n\n{test_problem}<end_of_turn>\n<start_of_turn>model\n<think>\n"

model.eval()
enc = tokenizer(test_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=384, temperature=0.6, pad_token_id=tokenizer.eos_token_id)

print("=== Post-AntiSD Gemma 4 Reasoning Trace ===")
print(tokenizer.decode(out[0], skip_special_tokens=False))

In [ ]:
# 9. Save and Push LoRA Adapter to Hugging Face Hub
# model.push_to_hub("your-username/gemma-4-e2b-it-antisd")
# tokenizer.push_to_hub("your-username/gemma-4-e2b-it-antisd")
model.save_pretrained("./gemma4_antisd_adapter")
tokenizer.save_pretrained("./gemma4_antisd_adapter")
print("Saved adapter locally to ./gemma4_antisd_adapter")